In [ ]:
!pip install gradio pandas numpy scikit-learn matplotlib


In [1]:
"""
Telco Customer Churn — single-file Gradio app.

Run:
    pip install gradio pandas numpy scikit-learn plotly
    python app.py

Tab order (workflow):
    1. Dataset            — upload + clean your CSV, see a quick profile.
    2. 3D Explorer         — interactive 3D scatter of tenure/charges/churn.
    3. Segment Treemap     — customer segments sized by count, colored by churn rate.
    4. Churn Flow          — parallel-categories flow from plan choices to churn.
    5. Train Model         — trains on the dataset loaded in tab 1.
    6. Prediction Dashboard — single-customer + batch scoring.

Works with the standard Kaggle IBM Telco schema AND the IBM sample schema
("Senior Citizen" as Yes/No, "Tenure Months", "Churn Label", geo / CLTV /
Churn Score / Churn Reason columns) — those are normalized or dropped
automatically since they leak the outcome or differ only in formatting.
"""

import re
import numpy as np
import pandas as pd
import gradio as gr
import plotly.express as px
import plotly.graph_objects as go

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix,
)

PLOTLY_TEMPLATE = "plotly_dark"
ACCENT_SCALE = ["#4C72B0", "#55A868", "#DD8452", "#C44E52"]

# ---------------------------------------------------------------------------
# Column handling: canonical feature names + aliases so both the classic
# Kaggle Telco schema and the IBM sample schema (spaced names, extra
# geo/outcome columns) both work without any manual renaming.
# ---------------------------------------------------------------------------

NUMERIC_FEATURES = ["tenure", "MonthlyCharges", "TotalCharges"]
CATEGORICAL_FEATURES = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService",
    "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod",
]
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# canonical -> list of normalized aliases (normalized = lowercase, no spaces/underscores)
ALIASES = {
    "tenure": ["tenure", "tenuremonths"],
    "MonthlyCharges": ["monthlycharges"],
    "TotalCharges": ["totalcharges"],
    "gender": ["gender"],
    "SeniorCitizen": ["seniorcitizen"],
    "Partner": ["partner"],
    "Dependents": ["dependents"],
    "PhoneService": ["phoneservice"],
    "MultipleLines": ["multiplelines"],
    "InternetService": ["internetservice"],
    "OnlineSecurity": ["onlinesecurity"],
    "OnlineBackup": ["onlinebackup"],
    "DeviceProtection": ["deviceprotection"],
    "TechSupport": ["techsupport"],
    "StreamingTV": ["streamingtv"],
    "StreamingMovies": ["streamingmovies"],
    "Contract": ["contract"],
    "PaperlessBilling": ["paperlessbilling"],
    "PaymentMethod": ["paymentmethod"],
    "Churn": ["churn", "churnlabel"],
}

# Columns that leak the outcome or are pure identifiers/geo — always dropped
# even if present, so the model can't cheat.
DROP_COLS_NORMALIZED = {
    "customerid", "count", "country", "state", "city", "zipcode", "latlong",
    "latitude", "longitude", "churnscore", "cltv", "churnreason", "churnvalue",
}


def _normalize(name: str) -> str:
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


def rename_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Rename whatever columns are present to our canonical names using
    the alias table; leave unrecognized columns untouched (they'll simply
    be ignored later since we only select ALL_FEATURES / Churn)."""
    norm_to_canonical = {}
    for canonical, aliases in ALIASES.items():
        for a in aliases:
            norm_to_canonical[a] = canonical

    rename_map = {}
    for col in df.columns:
        key = _normalize(col)
        if key in norm_to_canonical:
            rename_map[col] = norm_to_canonical[key]
    return df.rename(columns=rename_map)


def _normalize_senior_citizen(series: pd.Series) -> pd.Series:
    """Kaggle schema uses 0/1, the IBM sample schema uses Yes/No — collapse
    both onto 0/1 so the encoder never sees both spellings for one concept."""
    def _map(v):
        s = str(v).strip().lower()
        if s in {"1", "1.0", "yes", "true"}:
            return 1
        if s in {"0", "0.0", "no", "false"}:
            return 0
        return np.nan
    return series.map(_map)


def clean_dataframe(df: pd.DataFrame, drop_duplicate_rows: bool = True) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]
    df = rename_columns(df)

    # Drop identifier / geo / outcome-leaking columns if present
    drop_cols = [c for c in df.columns if _normalize(c) in DROP_COLS_NORMALIZED]
    df = df.drop(columns=drop_cols, errors="ignore")

    # Safety net: if any rename collision left duplicate column names, keep the first
    df = df.loc[:, ~df.columns.duplicated()]

    if "TotalCharges" in df.columns:
        df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
        # Known Telco-data quirk: brand-new customers (tenure == 0) sometimes
        # have a blank TotalCharges instead of 0 — fill with 0 rather than
        # letting the imputer guess a mid-range value for them.
        if "tenure" in df.columns:
            zero_tenure_blank = df["TotalCharges"].isna() & (pd.to_numeric(df["tenure"], errors="coerce") == 0)
            df.loc[zero_tenure_blank, "TotalCharges"] = 0.0

    if "SeniorCitizen" in df.columns:
        df["SeniorCitizen"] = _normalize_senior_citizen(df["SeniorCitizen"])

    obj_cols = df.select_dtypes(include="object").columns
    for c in obj_cols:
        df[c] = df[c].astype(str).str.strip()

    if drop_duplicate_rows:
        df = df.drop_duplicates()
    return df


def prepare_xy(df: pd.DataFrame):
    df = df.copy()
    for col in ALL_FEATURES:
        if col not in df.columns:
            df[col] = np.nan
    X = df[ALL_FEATURES].copy()

    y = None
    if "Churn" in df.columns:
        raw = df["Churn"].astype(str).str.strip().str.lower()
        y = raw.map({"yes": 1, "no": 0, "1": 1, "0": 0, "1.0": 1, "0.0": 0})
        if y.isna().any():
            y = y.fillna(pd.to_numeric(df["Churn"], errors="coerce"))
    return X, y


def build_pipeline() -> Pipeline:
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric_pipe, NUMERIC_FEATURES),
        ("cat", categorical_pipe, CATEGORICAL_FEATURES),
    ])
    model = RandomForestClassifier(
        n_estimators=300, max_depth=10, min_samples_leaf=3,
        class_weight="balanced", random_state=42, n_jobs=-1,
    )
    return Pipeline([("preprocessor", preprocessor), ("model", model)])


def oov_rate(df: pd.DataFrame) -> float:
    """Share of categorical cells whose value was never seen during training —
    the #1 real-world cause of batch predictions that all land on one side.
    High OOV means the uploaded file's schema doesn't match what the model
    was trained on (different spellings, a different export format, etc.)."""
    if STATE["train_categories"] is None:
        return 0.0
    total, unseen = 0, 0
    for col, seen_vals in STATE["train_categories"].items():
        if col not in df.columns:
            continue
        vals = df[col].astype(str).str.strip()
        total += len(vals)
        unseen += (~vals.isin(seen_vals)).sum()
    return (unseen / total) if total else 0.0


# ---------------------------------------------------------------------------
# Global in-memory state (single-user local app — no need for a DB)
# ---------------------------------------------------------------------------

STATE = {
    "df": None,               # cleaned dataset loaded in the Dataset tab
    "pipeline": None,
    "feature_names": None,
    "importances": None,
    "metrics": None,
    "train_categories": None,  # col -> set of category values seen at fit time
}


# ---------------------------------------------------------------------------
# Tab 1: Dataset
# ---------------------------------------------------------------------------

def load_dataset(file):
    empty_fig = go.Figure().update_layout(template=PLOTLY_TEMPLATE)
    if file is None:
        return ("Upload a CSV to get started.", None,
                empty_fig, empty_fig, empty_fig)

    raw = pd.read_csv(file.name)
    df = clean_dataframe(raw)
    STATE["df"] = df

    n_rows, n_cols = df.shape
    missing = df.isna().sum()
    missing = missing[missing > 0]

    lines = [f"### Dataset loaded ✅", f"**Rows:** {n_rows}  |  **Columns used by the model:** {len(ALL_FEATURES)}"]
    if len(missing):
        lines.append("**Missing values:** " + ", ".join(f"{c}: {v}" for c, v in missing.items()))
    else:
        lines.append("**Missing values:** none detected")

    if "Churn" in df.columns:
        counts = df["Churn"].astype(str).str.strip().value_counts()
        lines.append("**Churn distribution:** " + ", ".join(f"{k}: {v}" for k, v in counts.items()))
    else:
        lines.append("**Note:** no Churn column found — you can still explore, but training needs one.")
    summary_md = "\n\n".join(lines)

    fig_3d = make_3d_scatter(df)
    fig_treemap = make_treemap(df)
    fig_flow = make_churn_flow(df)

    return summary_md, df.head(15), fig_3d, fig_treemap, fig_flow


# ---------------------------------------------------------------------------
# Tab 2: 3D Explorer
# ---------------------------------------------------------------------------

def make_3d_scatter(df: pd.DataFrame):
    d = df.dropna(subset=["tenure", "MonthlyCharges", "TotalCharges"]).copy()
    if d.empty:
        return go.Figure().update_layout(template=PLOTLY_TEMPLATE, title="No numeric data to plot")

    color_col = "Churn" if "Churn" in d.columns else ("Contract" if "Contract" in d.columns else None)
    fig = px.scatter_3d(
        d, x="tenure", y="MonthlyCharges", z="TotalCharges",
        color=color_col, opacity=0.75, size_max=8,
        color_discrete_sequence=ACCENT_SCALE,
        template=PLOTLY_TEMPLATE,
        title="Customers in tenure / monthly charge / total charge space",
    )
    fig.update_traces(marker=dict(size=4, line=dict(width=0)))
    fig.update_layout(
        scene=dict(
            xaxis_title="Tenure (months)",
            yaxis_title="Monthly Charges ($)",
            zaxis_title="Total Charges ($)",
        ),
        margin=dict(l=0, r=0, t=40, b=0),
        height=560,
    )
    return fig


# ---------------------------------------------------------------------------
# Tab 3: Segment Treemap
# ---------------------------------------------------------------------------

def make_treemap(df: pd.DataFrame):
    needed = ["Contract", "InternetService", "PaymentMethod"]
    if not all(c in df.columns for c in needed):
        return go.Figure().update_layout(template=PLOTLY_TEMPLATE, title="Missing columns for treemap")

    d = df.copy()
    d["_count"] = 1
    if "Churn" in d.columns:
        d["_churn_num"] = d["Churn"].astype(str).str.strip().str.lower().map(
            {"yes": 1, "no": 0, "1": 1, "0": 0}
        )
        color_kwargs = dict(color="_churn_num", color_continuous_scale="RdYlGn_r",
                             range_color=[0, 1])
        title = "Customer segments — size = customer count, color = churn rate"
    else:
        color_kwargs = dict(color="_count", color_continuous_scale="Blues")
        title = "Customer segments — size and color = customer count"

    fig = px.treemap(
        d, path=[px.Constant("All customers"), "Contract", "InternetService", "PaymentMethod"],
        values="_count", template=PLOTLY_TEMPLATE, title=title, **color_kwargs,
    )
    fig.update_layout(margin=dict(l=0, r=0, t=40, b=0), height=560)
    return fig


# ---------------------------------------------------------------------------
# Tab 4: Churn Flow (parallel categories)
# ---------------------------------------------------------------------------

def make_churn_flow(df: pd.DataFrame):
    needed = ["Contract", "InternetService", "PaperlessBilling"]
    if not all(c in df.columns for c in needed) or "Churn" not in df.columns:
        return go.Figure().update_layout(template=PLOTLY_TEMPLATE, title="Need Contract / InternetService / PaperlessBilling / Churn columns")

    d = df.copy()
    d["ChurnNum"] = d["Churn"].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0, "1": 1, "0": 0})
    d = d.dropna(subset=["ChurnNum"])

    fig = px.parallel_categories(
        d, dimensions=["Contract", "InternetService", "PaperlessBilling", "Churn"],
        color="ChurnNum", color_continuous_scale="RdYlGn_r",
        template=PLOTLY_TEMPLATE,
        title="How plan choices flow into churn",
    )
    fig.update_layout(margin=dict(l=40, r=40, t=40, b=20), height=520)
    return fig


# ---------------------------------------------------------------------------
# Tab 5: Train Model
# ---------------------------------------------------------------------------

def train_model(override_file):
    if override_file is not None:
        raw = pd.read_csv(override_file.name)
        df = clean_dataframe(raw)
        STATE["df"] = df
    else:
        df = STATE["df"]

    if df is None:
        return "Load a dataset in the **Dataset** tab first (or upload one here to override it).", None, None

    X, y = prepare_xy(df)
    if y is None or y.isna().all():
        return (
            "No usable 'Churn' column found (expected 'Churn', 'Churn Label', "
            "or 'Churn Value'). Check your CSV.", None, None,
        )

    mask = ~y.isna()
    X, y = X[mask], y[mask].astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    pipeline = build_pipeline()
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba),
    }
    cm = confusion_matrix(y_test, y_pred)

    feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
    importances = pd.Series(
        pipeline.named_steps["model"].feature_importances_, index=feature_names
    ).sort_values(ascending=False)

    # Remember which categorical values the model actually trained on, so the
    # Prediction tab can flag batch files whose schema doesn't match.
    train_categories = {}
    for col in CATEGORICAL_FEATURES:
        train_categories[col] = set(X_train[col].astype(str).str.strip().unique())

    STATE["pipeline"] = pipeline
    STATE["feature_names"] = feature_names
    STATE["importances"] = importances
    STATE["metrics"] = metrics
    STATE["train_categories"] = train_categories

    summary = f"""### Model trained ✅

Trained on **{len(X_train)}** customers, tested on **{len(X_test)}** held-out customers.
Churn rate in data: **{y.mean():.1%}**

| Metric | Value |
|---|---|
| Accuracy | {metrics['accuracy']:.3f} |
| Precision | {metrics['precision']:.3f} |
| Recall | {metrics['recall']:.3f} |
| F1 score | {metrics['f1']:.3f} |
| ROC AUC | {metrics['roc_auc']:.3f} |
"""

    fig_cm = go.Figure(data=go.Heatmap(
        z=cm, x=["Predicted No", "Predicted Yes"], y=["Actual No", "Actual Yes"],
        colorscale="Blues", showscale=False, text=cm, texttemplate="%{text}",
        textfont=dict(size=16),
    ))
    fig_cm.update_layout(template=PLOTLY_TEMPLATE, title="Confusion Matrix", height=380,
                          margin=dict(l=10, r=10, t=40, b=10))

    top = importances.head(12)[::-1]
    fig_imp = go.Figure(go.Bar(x=top.values, y=top.index, orientation="h", marker_color="#4C72B0"))
    fig_imp.update_layout(template=PLOTLY_TEMPLATE, title="Top Feature Importances", height=420,
                           margin=dict(l=10, r=10, t=40, b=10))

    return summary, fig_cm, fig_imp


# ---------------------------------------------------------------------------
# Tab 6: Prediction Dashboard
# ---------------------------------------------------------------------------

YES_NO = ["Yes", "No"]
GENDER = ["Female", "Male"]
INTERNET_SERVICE = ["DSL", "Fiber optic", "No"]
MULTI_LINES = ["Yes", "No", "No phone service"]
INTERNET_DEPENDENT = ["Yes", "No", "No internet service"]
CONTRACT = ["Month-to-month", "One year", "Two year"]
PAYMENT = ["Electronic check", "Mailed check", "Bank transfer (automatic)", "Credit card (automatic)"]


def predict_single(gender, senior_citizen, partner, dependents, tenure,
                    phone_service, multiple_lines, internet_service,
                    online_security, online_backup, device_protection, tech_support,
                    streaming_tv, streaming_movies, contract, paperless_billing,
                    payment_method, monthly_charges, total_charges, threshold):
    if STATE["pipeline"] is None:
        return "Train the model first (Train Model tab).", None

    row = pd.DataFrame([{
        "gender": gender, "SeniorCitizen": senior_citizen, "Partner": partner,
        "Dependents": dependents, "tenure": tenure, "PhoneService": phone_service,
        "MultipleLines": multiple_lines, "InternetService": internet_service,
        "OnlineSecurity": online_security, "OnlineBackup": online_backup,
        "DeviceProtection": device_protection, "TechSupport": tech_support,
        "StreamingTV": streaming_tv, "StreamingMovies": streaming_movies,
        "Contract": contract, "PaperlessBilling": paperless_billing,
        "PaymentMethod": payment_method, "MonthlyCharges": monthly_charges,
        "TotalCharges": total_charges,
    }])[ALL_FEATURES]
    row["SeniorCitizen"] = _normalize_senior_citizen(row["SeniorCitizen"])

    pipeline = STATE["pipeline"]
    proba = pipeline.predict_proba(row)[0, 1]
    label = "⚠️ Likely to churn" if proba >= threshold else "✅ Likely to stay"
    result_md = f"### {label}\n\n**Churn probability: {proba:.1%}**  (threshold: {threshold:.0%})"

    top = STATE["importances"].head(10)[::-1]
    fig = go.Figure(go.Bar(x=top.values, y=top.index, orientation="h", marker_color="#4C72B0"))
    fig.update_layout(template=PLOTLY_TEMPLATE, title="Overall top features driving this model's predictions",
                       height=420, margin=dict(l=10, r=10, t=40, b=10))

    return result_md, fig


def predict_batch(file, threshold):
    empty_fig = go.Figure().update_layout(template=PLOTLY_TEMPLATE)
    if STATE["pipeline"] is None:
        return "Train the model first (Train Model tab).", None, None, empty_fig
    if file is None:
        return "Upload a CSV to run batch predictions.", None, None, empty_fig

    raw = pd.read_csv(file.name)
    df = clean_dataframe(raw, drop_duplicate_rows=False)
    X, _ = prepare_xy(df)

    proba = STATE["pipeline"].predict_proba(X)[:, 1]
    result = raw.copy()
    result["Churn_Probability"] = proba
    result["Churn_Prediction"] = np.where(proba >= threshold, "Yes", "No")

    out_path = "batch_predictions.csv"
    result.to_csv(out_path, index=False)

    pct_yes = (result["Churn_Prediction"] == "Yes").mean()
    summary_lines = [
        f"Scored **{len(result)}** customers at a **{threshold:.0%}** decision threshold. "
        f"Predicted churners: **{(result['Churn_Prediction'] == 'Yes').sum()}** ({pct_yes:.1%})."
    ]

    oov = oov_rate(X)
    if oov > 0.15:
        summary_lines.append(
            f"⚠️ **{oov:.0%} of categorical values in this file weren't seen during training** "
            "— that usually means this file's column headers or category spellings "
            "(e.g. `Senior Citizen: Yes/No` vs `SeniorCitizen: 0/1`) don't match the "
            "dataset the model was trained on. Mismatched values get treated as "
            "'unknown', which pushes every prediction toward the same class. "
            "Re-train on this file's schema, or fix the header/value formatting to match."
        )
    if pct_yes in (0.0, 1.0):
        summary_lines.append(
            "⚠️ Every row landed on the same side of the threshold. Check the probability "
            "distribution below — if it's a narrow spike rather than a spread, that's the "
            "schema-mismatch symptom above; if it's a genuine wide spread just below/above "
            f"{threshold:.0%}, try moving the threshold slider."
        )

    fig_hist = go.Figure(go.Histogram(x=proba, nbinsx=30, marker_color="#4C72B0"))
    fig_hist.add_vline(x=threshold, line_dash="dash", line_color="#DD8452",
                        annotation_text="threshold")
    fig_hist.update_layout(template=PLOTLY_TEMPLATE, title="Predicted churn probability — distribution",
                            xaxis_title="Churn probability", yaxis_title="Customers",
                            height=360, margin=dict(l=10, r=10, t=40, b=10))

    return "\n\n".join(summary_lines), result.head(20), out_path, fig_hist


# ---------------------------------------------------------------------------
# Gradio Blocks UI
# ---------------------------------------------------------------------------

with gr.Blocks(title="Telco Customer Churn", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📉 Telco Customer Churn App")
    gr.Markdown("Upload your dataset once, explore it, then train and predict.")

    with gr.Tab("1. Dataset"):
        gr.Markdown("Upload your Telco CSV. This dataset feeds every tab below.")
        dataset_file = gr.File(label="Dataset CSV", file_types=[".csv"])
        dataset_button = gr.Button("Load Dataset", variant="primary")
        dataset_summary = gr.Markdown()
        dataset_preview = gr.Dataframe(label="Preview (first 15 rows)")

    with gr.Tab("2. 3D Explorer"):
        gr.Markdown("Rotate and zoom — tenure vs. monthly charges vs. total charges.")
        plot_3d = gr.Plot()

    with gr.Tab("3. Segment Treemap"):
        gr.Markdown("Contract → Internet Service → Payment Method, sized by customer count.")
        plot_treemap = gr.Plot()

    with gr.Tab("4. Churn Flow"):
        gr.Markdown("How plan choices flow into churn outcomes.")
        plot_flow = gr.Plot()

    dataset_button.click(
        load_dataset, inputs=[dataset_file],
        outputs=[dataset_summary, dataset_preview, plot_3d, plot_treemap, plot_flow],
    )

    with gr.Tab("5. Train Model"):
        gr.Markdown("Trains on the dataset loaded in the **Dataset** tab. "
                     "Optionally upload a different CSV below to override it.")
        train_file = gr.File(label="Override training CSV (optional)", file_types=[".csv"])
        train_button = gr.Button("Train Model", variant="primary")
        train_summary = gr.Markdown()
        with gr.Row():
            cm_plot = gr.Plot(label="Confusion Matrix")
            imp_plot = gr.Plot(label="Feature Importances")
        train_button.click(train_model, inputs=[train_file], outputs=[train_summary, cm_plot, imp_plot])

    with gr.Tab("6. Prediction Dashboard"):
        threshold_slider = gr.Slider(0.05, 0.95, value=0.5, step=0.05,
                                      label="Churn decision threshold",
                                      info="Lower it to flag more customers as at-risk; raise it to flag fewer.")

        gr.Markdown("### Single Customer Prediction")
        with gr.Row():
            with gr.Column():
                gender = gr.Dropdown(GENDER, value="Female", label="Gender")
                senior_citizen = gr.Dropdown(YES_NO, value="No", label="Senior Citizen")
                partner = gr.Dropdown(YES_NO, value="No", label="Has Partner")
                dependents = gr.Dropdown(YES_NO, value="No", label="Has Dependents")
                tenure = gr.Slider(0, 72, value=12, step=1, label="Tenure (months)")
                phone_service = gr.Dropdown(YES_NO, value="Yes", label="Phone Service")
                multiple_lines = gr.Dropdown(MULTI_LINES, value="No", label="Multiple Lines")
            with gr.Column():
                internet_service = gr.Dropdown(INTERNET_SERVICE, value="Fiber optic", label="Internet Service")
                online_security = gr.Dropdown(INTERNET_DEPENDENT, value="No", label="Online Security")
                online_backup = gr.Dropdown(INTERNET_DEPENDENT, value="No", label="Online Backup")
                device_protection = gr.Dropdown(INTERNET_DEPENDENT, value="No", label="Device Protection")
                tech_support = gr.Dropdown(INTERNET_DEPENDENT, value="No", label="Tech Support")
                streaming_tv = gr.Dropdown(INTERNET_DEPENDENT, value="No", label="Streaming TV")
                streaming_movies = gr.Dropdown(INTERNET_DEPENDENT, value="No", label="Streaming Movies")
            with gr.Column():
                contract = gr.Dropdown(CONTRACT, value="Month-to-month", label="Contract")
                paperless_billing = gr.Dropdown(YES_NO, value="Yes", label="Paperless Billing")
                payment_method = gr.Dropdown(PAYMENT, value="Electronic check", label="Payment Method")
                monthly_charges = gr.Number(value=70.0, label="Monthly Charges ($)")
                total_charges = gr.Number(value=840.0, label="Total Charges ($)")

        predict_button = gr.Button("Predict Churn", variant="primary")
        predict_result = gr.Markdown()
        predict_plot = gr.Plot(label="Model Explanation")
        predict_button.click(
            predict_single,
            inputs=[gender, senior_citizen, partner, dependents, tenure,
                    phone_service, multiple_lines, internet_service,
                    online_security, online_backup, device_protection, tech_support,
                    streaming_tv, streaming_movies, contract, paperless_billing,
                    payment_method, monthly_charges, total_charges, threshold_slider],
            outputs=[predict_result, predict_plot],
        )

        gr.Markdown("---\n### Batch Prediction")
        batch_file = gr.File(label="Upload CSV", file_types=[".csv"])
        batch_button = gr.Button("Run Batch Prediction")
        batch_summary = gr.Markdown()
        batch_preview = gr.Dataframe(label="Preview (first 20 rows)")
        batch_hist = gr.Plot(label="Probability Distribution")
        batch_download = gr.File(label="Download full results")
        batch_button.click(predict_batch, inputs=[batch_file, threshold_slider],
                            outputs=[batch_summary, batch_preview, batch_download, batch_hist])

if __name__ == "__main__":
    demo.launch()

/tmp/ipykernel_2085/2978949868.py:540: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Telco Customer Churn", theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dd4513a342d4af821a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
